In [3]:
import pickle
import pandas as pd
import numpy as np

In [5]:
books = pd.read_csv("../data/processed/books_enhanced.csv")
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
user_item_matrix = pickle.load(
    open("../data/processed/user_item_matrix.pkl", "rb")
)

print("Processed data loaded.")

Processed data loaded.


In [6]:
similarity_matrix = pickle.load(
    open("../models/similarity_matrix.pkl", "rb")
)

reconstructed_df = pickle.load(
    open("../models/svd_model.pkl", "rb")
)

strong_rules = pickle.load(
    open("../models/association_rules_model.pkl", "rb")
)

In [7]:
def recommend_books_svd(user_id, top_n=20):

    if user_id not in reconstructed_df.index:
        return []

    user_ratings = reconstructed_df.loc[user_id]

    already_rated = user_item_matrix.loc[user_id] > 0

    recommendations = user_ratings[~already_rated] \
        .sort_values(ascending=False) \
        .head(top_n) \
        .index.tolist()

    return recommendations

In [8]:
def recommend_books_content(book_id, top_n=20):

    if book_id not in similarity_matrix.columns:
        return []

    scores = similarity_matrix[book_id].sort_values(ascending=False)

    return scores.iloc[1:top_n+1].index.tolist()

In [9]:
def recommend_for_user(user_books, rules, top_n=20):

    filtered = rules[
        rules['antecedents'].apply(
            lambda x: any(book in x for book in user_books)
        )
    ]

    if filtered.empty:
        return []

    filtered = filtered.copy()

    filtered['score'] = (
        0.4 * filtered['confidence'] +
        0.6 * filtered['lift']
    )

    filtered = filtered.sort_values('score', ascending=False)

    recommendations = []

    for cons in filtered['consequents']:
        recommendations.extend(list(cons))

    recommendations = list(set(recommendations) - set(user_books))

    return recommendations[:top_n]

In [10]:
print("recommend_books_svd" in dir())
print("recommend_books_content" in dir())
print("recommend_for_user" in dir())

True
True
True


In [12]:
print("Similarity columns sample:", similarity_matrix.columns[:5])
print("SVD columns sample:", reconstructed_df.columns[:5])

Similarity columns sample: Index([1, 2, 3, 4, 5], dtype='int64', name='goodreads_book_id')
SVD columns sample: Index([1, 2, 3, 4, 5], dtype='int64', name='book_id')


In [51]:
popularity_score = books.set_index("book_id")["ratings_count"]

In [61]:
def hybrid_recommend(user_id=None, book_id=None, top_n=10):

    # Cold start for new users
    if user_id is None or user_id not in user_item_matrix.index:
        return books.sort_values(
            "average_rating",
            ascending=False
        ).head(top_n)["book_id"].tolist()

    scores = {}

    CF_WEIGHT = 0.7
    CB_WEIGHT = 0.2
    AR_WEIGHT = 0.1

    # Collaborative filtering
    cf_recs = recommend_books_svd(user_id, top_n=20)
    for i, book in enumerate(cf_recs):
        scores[book] = scores.get(book, 0) + CF_WEIGHT * (1/(i+1))

    # Content-based
    if book_id is not None:
        cb_recs = recommend_books_content(book_id, top_n=20)
        for i, book in enumerate(cb_recs):
            scores[book] = scores.get(book, 0) + CB_WEIGHT * (1/(i+1))

    # Association rules
    user_books = user_item_matrix.loc[user_id]
    user_books = user_books[user_books > 0].index.tolist()

    ar_recs = recommend_for_user(user_books, strong_rules, top_n=20)
    for i, book in enumerate(ar_recs):
        scores[book] = scores.get(book, 0) + AR_WEIGHT * (1/(i+1))

    if not scores:
        return []

    # Popularity boost
    for book in scores:
        scores[book] += 0.05 * popularity_score.get(book, 0)
    
    # Sort by hybrid score
    sorted_books = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # Diversity filtering
    recommended = []
    for book, _ in sorted_books:
        if book not in recommended and book not in user_books:
            recommended.append(book)
        if len(recommended) >= top_n:
            break
    
    return recommended

In [63]:
popularity_score = books.set_index("book_id")["ratings_count"]

In [65]:
hybrid_recommend(user_id=10, book_id=5, top_n=5)

[1, 2, 3, 6, 7]

In [67]:
hybrid_recommend(
    user_id=reconstructed_df.index[0],
    book_id=similarity_matrix.columns[0],
    top_n=5
)

[2, 3, 5, 6, 8]

In [69]:
from sklearn.model_selection import train_test_split

ratings_train = []
ratings_test = []

for user in ratings['user_id'].unique():

    user_data = ratings[ratings['user_id'] == user]

    if len(user_data) > 5:
        train, test = train_test_split(
            user_data,
            test_size=0.2,
            random_state=42
        )

        ratings_train.append(train)
        ratings_test.append(test)

ratings_train = pd.concat(ratings_train)
ratings_test = pd.concat(ratings_test)

In [70]:
def precision_at_k_hybrid(user_id, k=20):

    if user_id not in ratings_test['user_id'].values:
        return None

    recommended_ids = hybrid_recommend(user_id=user_id, book_id=None, top_n=k)

    if not recommended_ids:
        return None

    relevant = ratings_test[
        ratings_test['user_id'] == user_id
    ]['book_id'].tolist()

    if not relevant:
        return None

    hits = len(set(recommended_ids) & set(relevant))

    return hits / k

In [71]:
precisions = []

for user in ratings_test['user_id'].unique():
    p = precision_at_k_hybrid(user, 20)
    if p is not None:
        precisions.append(p)

print("Hybrid Precision@20:", round(sum(precisions)/len(precisions), 3))

Hybrid Precision@20: 0.003


In [72]:
user = ratings_test['user_id'].unique()[0]

recommended = hybrid_recommend(user_id=10, top_n=10)

print(recommended)
print(type(recommended[0]))
print(type(ratings_test['book_id'].iloc[0]))

[7, 8, 9, 22, 43, 60, 76, 119, 100, 101]
<class 'int'>
<class 'numpy.int64'>


In [73]:
recommended = hybrid_recommend(user_id=10, top_n=10)



In [74]:
ratings_test[ratings_test['rating'] >= 3].shape

(193538, 4)

In [75]:
user = ratings_test['user_id'].unique()[0]

recommended = hybrid_recommend(user_id=user, top_n=10)

relevant = ratings_test[
    (ratings_test['user_id'] == user) &
    (ratings_test['rating'] >= 3)
]['book_id'].tolist()

print("User:", user)
print("Recommended:", recommended)
print("Relevant:", relevant)
print("Intersection:", set(recommended) & set(relevant))

User: 1
Recommended: [5, 15, 26, 55, 63, 64, 80, 87, 83, 81]
Relevant: [2002, 60, 256, 496, 258, 4, 414, 2063, 45, 162, 119, 177, 140, 1796, 6665, 57, 115]
Intersection: set()
